# Vinho Verde Wine Quality Analysis Notebook

___

### *Notebook by: [**Sofia French**](https://www.ncl.ac.uk/learning-and-teaching/digital-technologies/nula/guidance-and-support/find-my-students/)*
### *Student No.: **`230057999`***
### *Module No.: **`2034`***
___

## Table of contents
___
#### 1. [Introduction](#1-overview)
#### 3. [The Problem Domain](#)
#### 2. [Setup](#2-setup) 
___

### Introduction

___

This notebook is heavily inspired by [Randal S. Olson's](https://nbviewer.org/github/rhiever/Data-Analysis-and-Machine-Learning-Projects/blob/master/example-data-science-notebook/Example%20Machine%20Learning%20Notebook.ipynb#Required-libraries) notebook, as mentioned in the references and report. Specifically, much of the structure has helped me effectively tell my "data science story" in a clean and effective way! :)

File structure

___

### The Problem Domain
___

As outlined eloquently in the task description, I have been *"hired to consult for a winery in Portugal"* who wish to understand the truth behind the **quality disparity** in their **Vinho-Verde** wines.

Provided are three datasets, 

```
• winequality-red.csv
• winequality-white.csv
• winequality.names
```

The two `.csv` files include large data for red and white wines, with **4899** and **1600** data points respectively, and each contains data points with a set of 12 variables; 11 of which are physicochemical, with the last being the customers perceived quality.

My task is to create *"one or more models to predict quality given all other variables"*, in order for the winery to determine which properties of their wine are most influential to quality.

___

### The Datasets at a Glance
___

#### Answering Olson's Questions 

[Randal S. Olson](https://nbviewer.org/github/rhiever/Data-Analysis-and-Machine-Learning-Projects/blob/master/example-data-science-notebook/Example%20Machine%20Learning%20Notebook.ipynb#Required-libraries) mentioned in his notebook, *"the first step to any data analysis project is to define the question or problem we're looking to solve, and to define a measure (or set of measures) for our success at solving that task"*. With that in mind, I examined the project and it's requirements using [Olson's](https://nbviewer.org/github/rhiever/Data-Analysis-and-Machine-Learning-Projects/blob/master/example-data-science-notebook/Example%20Machine%20Learning%20Notebook.ipynb#Required-libraries) questions:


> **Did you specify the type of data analytic question?**
I'm trying to predict the quality of Vinho Verde wines based on 11 properties, e.g. `acidity`, `sugar`, and `alcohol content`.
This is both an exploratory and predictive modelling problem. First, I'll need to explore and visualise how these features relate to a wines perceived quality. Then, I can build models to classify and regress the wine quality based on the most influential features.

> **Did you define the metric for success before beginning?**
It differs based on whether I'm using classification or regression, but both should use `k-fold cross validation`: in this project I will use **5-fold CV**.

For *classification* models:
1. F1-score
2. AUC
3. ROC
TBR

For *regression* models:
1. RMSE 
2. MAE
3. Related "error metrics"
TBR

For a model to be considered successful, it must outperform a base threshold (**the mean**) and generalise well across both red and white wines.

> **Did you understand the context for the question and the scientific or business application?**
I'm consulting for a winery in Portugal that produces Vinho Verde wines - red and white. They want to understand which factors most strongly determine wine quality so they can optimise the production process.

This analysis will inform the winery on which properties of wine should increase, decrease or remain constant during wine production in order to improve the quality of wines effectively (at least, I would assume, it's hard to get ahold of any stakeholders).

> **Did you record the experimental design?**
In both datasets, each record corresponds to a single wine. As mentioned in [The Problem Domain](#3-the-problem-domain), "the two datasets include 4899 and 1600 data points" where "each data point is a set of 12 variables; 11 of these are physicochemical measures, and the last is the perceived quality (sensory output)". 

> **Did you consider whether the question could be answered with the available data?**
The dataset is well suited for determining factors that influence wine quality. However, there are **limitations**:
- Quality is subjective to consumer
- There are likely other factors that influence quality that are not recorded

#### Exploring the Datasets

Before I begin with the required tasks, as suggested in [Olson's](https://nbviewer.org/github/rhiever/Data-Analysis-and-Machine-Learning-Projects/blob/master/example-data-science-notebook/Example%20Machine%20Learning%20Notebook.ipynb#Required-libraries) notebook and hinted at in the [Canvas](https://ncl.instructure.com/courses/54995/pages/task-description-for-data-science?module_item_id=3344525), it's best to quickly check the data's sanity.  

Specifically, I'm checking for errors or unexpected qualities in either dataset. For this project, both datasets are well regarded and tested, meaning it is safe to assume the data is clean. However, for good practice, it can't hurt to double check.

To do this, I'll read both datasets into a pandas data frame (*df*) and display the first 5 records. First, the red wine dataset:

In [ ]:
import pandas as pd

wine_red = pd.read_csv("../datasets/winequality-red.csv",sep=";")
wine_red.head()

As expected, the data is loaded and populated. Now for white wine: 

In [ ]:
wine_white = pd.read_csv("../datasets/winequality-white.csv",sep=";")
wine_white.head()

Again, the dataset appears complete, with 12 fields and each as listed in the brief. No problems here.

Next, starting again with red wine, I'll use pandas `describe` to examine two different, key fields: `alcohol` and `residual sugar`.

In [ ]:
wine_red[["alcohol", "residual sugar"]].describe()

Focusing first on `alcohol`:
- It has a good range, with the `min` value at `8.4` and `max` value at `14.9`.
- This makes it clear why later separating into a three-valued variable `alcohol_content`, with *low, mid, and high* values would be beneficial - there is sufficient spread to justify three levels.

Next on `residual sugar`:
- It appears most red wines are dry as the mean is approximately `3`.
- The data is skewed heavily to the right tail, as some outlier values having residual sugar as high as `15.5`.
- This makes it clear why a binary value splitting the sweetness evenly (`isSweet`) would be beneficial when examining this feature.

**Moving onto white wine**, I'll perform the same analysis.

In [ ]:
wine_white[["alcohol", "residual sugar"]].describe()

Focusing first on `alcohol`:
- Very similar to red wine, however white wines have a slightly higher mean (`10.51...` vs. `10.42...`).
- Spread is also larger than red (std as `1.23` vs. `1.07`).
- But again, this is a well distributed variable and thus supports separating into a three-valued `alcohol_content` variable.

Next on `residual sugar`:
- Much higher mean than in the red wine dataset (mean = `6.39...` vs. `2.54...`).
- The max value (`65.8`) is an extreme outlier.
- Again, the distribution is very right tail heavy, though much more so than with the red wines.
- The mean (`6.39...`) again suggests many dry wines and a lower portion of sweet wines.

#### First Conclusions

For both datasets:
- In the later analysis with the `alcohol_cat` category, it would make sense to split into three values as both datasets have good spread (between `min` and `max` valued.)
- In the later analysis with the `isSweet` category, it would make sense to split on the median value to avoid a lack of high sweetness wines.

#### First Plots

Before I explore the two fields any further, in preparation for both initial and later plots, I'll import my chosen libraries: `matplotlib` and `seaborn`. 

Both were chosen over the other mentioned alternative, `Altair` for the same reasons:
1. Simpler
2. More online tutorials
3. More common

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import seaborn as sb

# Set style
sb.set_theme(style="whitegrid", font_scale=1.2)
palette_red = sb.color_palette("Reds", n_colors=9)
palette_white = sb.color_palette("Blues", n_colors=9)

Next, I generated a scatterplot matrix. 

This was suggested in [Olson's](https://nbviewer.org/github/rhiever/Data-Analysis-and-Machine-Learning-Projects/blob/master/example-data-science-notebook/Example%20Machine%20Learning%20Notebook.ipynb#Required-libraries) notebook as a tool to quickly check for errors or anomalies.

Originally, I compared every category against `quality`, but the 121 generated scatterplots were too busy for me to gain any insight from. Instead, for a quick initial look, I compare `alcohol`, `volatile acidity`, `sulphates`, `citric acid` against eachother with the colours of the points representing `quality` from a scale, for reds, of 3 -> 8.

I saved all the generated figures for this project in the directory `figures/`. For more information, see [Introduction](#1-introduction) where I explain the file structure.

In [ ]:
init_scatterplot_red = sb.pairplot(wine_red[["alcohol", "volatile acidity", "sulphates", "citric acid", "quality"]], hue="quality", palette=palette_red)
init_scatterplot_red.savefig("../figures/init_scatterplot_red")

#### Observations

A quick look at patterns in these plots suggests:

**For `Alcohol`**:
- In every plot (excluding `volatile acidity`) where `alcohol` is compared, higher quality wines (6 -> 8) tend to have higher alcohol content.
- The KDE plot shows that low quality wines peak at **low alcohol levels**, whereas high quality wines peak around **12**

**For `Volatile Acidity`**
- The trend is the opposite to `alcohol`. Higher `volatile acidity` appears to cause lower quality wine.
- Wines with qualities 3 -> 4 cluster at higher values of `volatile acidity`.

**For `Sulphates`**
- There's a slight pattern with high quality wines having slightly higher sulphates, but not strongly enough to apply causation based on the limited plots.

**For `Citric Acid`**
- Hard to interpret — there’s no obvious relationship and dots are scattered evenly.

#### First Plot Conclusions

- **Alcohol** appears to be strongly correlated with better quality wines, as is the most predictive feature out of the 4 I plotted and interpreted.
- **Volatile acidity** is negatively correlated with quality.
- **Sulphates** may be slightly positively correlated with quality.
- **Citric acid** may not be a strong predictor.

Now lets examine the ***white wine dataset***.

In [ ]:
init_scatterplot_white = sb.pairplot(wine_white[["alcohol", "volatile acidity", "sulphates", "citric acid", "quality"]], hue="quality", palette=palette_white)
init_scatterplot_white.savefig("../figures/init_scatterplot_white")

#### Observations

A quick look at patterns here suggests:

**For `Alcohol`**:
- Just like in red wine: higher alcohol tends to mean higher quality.
- The KDE plot shows highest quality 8 -> 9 wines peak again around the value **12**.

**For `Volatile Acidity`**
- Less dramatic than in red wine, but still: higher volatile acidity tends to mean lower quality.
- Quality 3 -> 5 values are slightly more concentrated at higher values.

**For `Sulphates`**
- Again, there's some separation: quality 8 -> 9 wines tend to have slightly higher sulphates.
- Distribution is tighter than in red wine.

**For `Citric Acid`**
- Similar to red: no obvious relationship with quality.
- KDE shows distributions are nearly overlapping across all quality levels.

#### First Plot Overall Conclusions

- **Alcohol** is a strong predictor of quality in both red and white wines.
- **Volatile acidity** is weakly negatively correlated with quality — maybe not as useful as in red wine.
- **Sulphates** again may have slight positive association, but not dramatically.
- **Citric acid** are probably not worth prioritising for later modelling.

## Setup

To begin the project, I loaded the **red and white wine datasets** (from their CSV files in directory `datasets/`), which contain properties and quality scores for different wines.

I also used plot styling with Seaborn to ensure clean visuals and easy plot readability/interprebility throughout the notebook.

In [ ]:
# # Load data
# wine_red.columns = wine_red.columns.str.strip()
# wine_white.columns = wine_white.columns.str.strip()

## Task 1A: Exploring the Datasets

Below is a side-by-side plot comparing quality scores for both datasets. The white wine dataset (4,898 samples) is more than three times the size of the red wine dataset (1,599 samples) and therefore has a higher y-axis.

**Comparison:**
- Both wines have median scores of around **5 and 6**.
- **Red wine** is **highly focused** around these mid-scores, with almost no wines above score 7 (narrower spread).
- **White wine** quality has a **wider range**, with more highly ranking wines (7–9) and a longer right tail (more dispersed).


In [ ]:
# Plots

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Red plot
sb.countplot(data=wine_red, x="quality", hue="quality",ax=axes[0],palette="Reds_r",legend=False)
axes[0].set_title("Red Wine Quality",fontsize=14)
axes[0].set_xlabel("Quality Score")
axes[0].set_ylabel("Count")

# White plot
sb.countplot(data=wine_white, x="quality",hue="quality",ax=axes[1],palette="Blues",legend=False)
axes[1].set_title("White Wine Quality",fontsize=14)
axes[1].set_xlabel("Quality Score")
axes[1].set_ylabel("Count")

# Both plot
plt.suptitle("Quality Distributions: Red vs White Wine", fontsize=16, weight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig("../figures/wine_quality_comparison.png", dpi=300)
plt.show()

### Red Wine Quality Distribution

This plot shows red wine quality ratings independently.

**Quality Plot Analysis:**
- Most red wine qualities are rated either **5 or 6**, with not many wines outside of this range.
- Few wines exist with quality scores of **3, 4, 7, or 8**, and there are none with a score of **9**.
- The distribution is **unimodal** and suggests **low quality variance**.

In [ ]:
# Red wine plot
plt.figure(figsize=(8, 5))
sb.countplot(data=wine_red, x="quality", hue="quality", palette="Reds_r", legend=False)
plt.title("Red Wine Quality Distribution", fontsize=14)
plt.xlabel("Quality Score")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("../figures/red_wine_quality.png", dpi=300)
plt.show()

### White Wine Quality Distribution

The white wine dataset, when plotted against quality, is considerably different.

**Quality Plot Analysis:**
- The mode is the same as red wine, at **6**, but the distribution is **broader** and includes higher ratings including **9**, unlike the red wine plot.
- White wines rated **7 or 8** are not uncommon, and a few wines score at **9**.
- Lower scores (**3, 4**) are also more frequent than in the red wines dataset.

In [ ]:
# White wine plot
plt.figure(figsize=(8, 5))
sb.countplot(data=wine_white, x="quality", hue="quality", palette="Blues", legend=False)
plt.title("White Wine Quality Distribution", fontsize=14)
plt.xlabel("Quality Score")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("../figures/white_wine_quality.png", dpi=300)
plt.show()

### End: Task 1A

### Summary

- Red wine quality is tightly clustered around scores 5 and 6.
- White wine quality has a larger spread, with more highly-rated samples (7 and 8), and a longer right tail.
- There are over 3x as many white wine samples as there are red.

## Task 1B/1C

### Discretise Alcohol Content Vars. Based on Distribution

To investigate how alcohol content influenced wine quality, I discretised alcohol content separately on whites and reds into a three valued variable (`alcohol_cat`) based on distribution:

- **low**: alcohol < (mean - stddev)
- **mid**: (mean - stddev) < alcohol < (mean + stddev)
- **high**: alcohol > (mean + stddev)

To do this, I made a helper function to calculate the dataset’s mean and standard deviation (`stddev`), and assigned each sample to one of the three categories.

In [ ]:
def add_alcohol_category(df):
    mean = df["alcohol"].mean()
    stddev = df["alcohol"].std()
    
    def categorise(alc):
        if alc < mean - stddev:
            return "low"
        elif alc > mean + stddev:
            return "high"
        else:
            return "mid"
    
    return df.assign(alcohol_cat=df["alcohol"].apply(categorise))

### Generating Red Wine Quality Plot *by Alcohol Category*

The boxplot below displays the relationship between red wine quality and `alcohol_cat` (low, mid, high).

**Observations:**
- A clear positive trend is shown across alcohol categories: this suggests **higher alcohol content is associated with higher quality scores**.
- The **median quality** increases from 5 (low) to 6 (mid), and 7 (high).
- The **spread** of quality scores *narrows* for the high alcohol group, indicating that there is higher consistent quality among red wines with higher alcohol content.
- Red wines in the low alcohol cat rarely achieve ratings above 6, and show more low outliers.

**Considerations:**
- Alcohol seems to be influential to the quality of the red wines.
- Thus the category `alcohol_cat` provides **meaningful stratification/classification** of quality and should be considered a possible feature in later predictive modelling.

In [ ]:
# Apply to both datasets
wine_red = add_alcohol_category(wine_red)
wine_white = add_alcohol_category(wine_white)

# Red wine
plt.figure(figsize=(10, 5))
sb.boxplot(
    data=wine_red,
    x="alcohol_cat",
    y="quality",
    palette="Reds",
    hue="alcohol_cat",
    order=["low", "mid", "high"],
    hue_order=["low", "mid", "high"]
)
plt.title("Red Wine Quality by alcohol_cat")
plt.savefig("../figures/red_quality_by_alcohol.png")
plt.show()

### Generating White Wine Quality Plot *by Alcohol Category*

The boxplot below displays the relationship between white wine quality and `alcohol_cat` (low, mid, high).

**Observations:**
- The same **positive correlation** seen in red wine also appears in white wine -> higher alcohol levels seem to map to higher quality.
- The **high alcohol wines** have the highest median quality and includes most of the highest outlier quality wines (7–9).
- Low alcohol wines are more tightly clustered around scores of 5–6, with less high-quality outliers.
- Compared to red wine, the differences are slightly less dramatic, but still consistent.

**Considerations:**
- Alcohol content is again **influential to quality**.
- This consistent trend across both wine types suggests the `alcohol_cat` variable can be used influentially for both datasets, making it a valuable variable for model training across datasets later.

In [ ]:
# White wine
plt.figure(figsize=(10, 5))
sb.boxplot(
    data=wine_white,
    x="alcohol_cat",
    y="quality",
    palette="Blues",
    hue="alcohol_cat",
    order=["low", "mid", "high"],
    hue_order=["low", "mid", "high"]
)
plt.title("White Wine Quality by alcohol_cat")
plt.savefig("../figures/white_quality_by_alcohol.png")
plt.show()

### End: Task 1B/1C

### Summary

In Task 1B/1C, I plotted quality against alcohol content, based on distribution, and observed and analysed how it related to wine quality across both red and white wines.

To do this, I made a new variable, `alcohol_cat`, by discretising the `alcohol` variable into three levels (low, mid, high) using each dataset’s mean and standard deviation.

**Key Observations:**

- For both red and white wines, **higher alcohol content correlates to higher quality**.
- The correlation is especially clear in red wines, where low alcohol wines rarely have higher quality ratings then 6, while high alcohol wines frequently have ratings of 7+.
- White wines show a similar, though less dramatic, trend — this means alcohol value can be used as a general predictor of quality.

**For Later Modelling:**

- The consistant correlaton of alcohol content to quality suggests this feature can be **highly useful in both regression and classification models**.
- Using a discretised variable, such as `alcohol_cat`, instead of only alcohol values, also improves interpretability.

## Task 1D/1E

### Plot Residual Sugar

The distribution of residual sugar is **skewed** for both red and white wine datasets, so to examine it's affect of quality, like with alcohol, I made a new binary variable `isSweet`.

`isSweet`, is:
- `1` -> if residual sugar >= median,
- `0` -> else (so residual sugar will be <= median).

Using the **median** means there will be an almost even split between sweet and dry, providing **balanced groups** with approximately the same amound of records.

In [ ]:
def add_is_sweet(df):
    threshold = df["residual sugar"].median()
    return df.assign(isSweet=(df["residual sugar"] >= threshold).astype(int))

# Apply to both datasets
wine_red = add_is_sweet(wine_red)
wine_white = add_is_sweet(wine_white)

### Red Wine: Quality by Sweetness

The boxplot below shows how quality varies between dry (`isSweet = 0`) and sweet (`isSweet = 1`) red wines.

**Observations:**
- The **median quality is the same** for both groups (quality = 6).
- Residual sugar *does not appear* to be influential in the quality of red wines.
- This suggests that **sweetness alone is not a reliable indicator of quality** for red wine, though it may have different affect when combined and compared with other categories (like alcohol and sweetness).

In [ ]:
# Red wine by sweetness
plt.figure(figsize=(10, 5))
sb.boxplot(data=wine_red, x="isSweet", y="quality", palette="Reds", hue="isSweet")
plt.title("Red Wine Quality by isSweet")
plt.savefig("../figures/red_quality_by_sugar.png")
plt.show()

### White Wine: Quality by Sweetness

The boxplot below shows how quality varies between dry (`isSweet = 0`) and sweet (`isSweet = 1`) white wines, and greatly differs from the red wine box plot.

**Observations:**
- In contrast to red wine, dry white wines (`isSweet = 0`) show a **higher median** and have higher qualities.
- Sweet white wines are much more tightly clustered and have less wines **exceeding a quality of 6**.
- This indicates a clearer trend: **dry white wines are more likely to be rated highly**.

In [ ]:
# White wine by sweetness
plt.figure(figsize=(10, 5))
sb.boxplot(data=wine_white, x="isSweet", y="quality", palette="Blues", hue="isSweet")
plt.title("White Wine Quality by isSweet")
plt.savefig("../figures/white_quality_by_sugar.png")
plt.show()

### End Task 1D/E

### Summary:

- Created an `isSweet` binary variable based on the median residual sugar for each wine type to guarantee an even split between dry and sweet wines in each dataset.
- Sweetness shows **little correlation with quality** in red wine, but a **clear negative trend** in white wine.

### For Later Modelling:
- `isSweet` may not be a useful indicator for red wine prediction.
- It may improve classification performance for white wines when used in combination with other categories.


## Task 2

### Correlation Matrix:

To determine which subset of variables would be most useful for learning, I made a correlation matrix to analyse relationships between:
- Each pair of variables
- Each variable and the outcome (`quality`)
- Plotted visually

I used the **Pearson correlation**, which measures **linear relationships** between continuous variables. I chose this metric because:
- All variables are numerical
- Most relationships with quality are linear

### Predictions:

Based on the previous plots, I predict:
- Alcohol content will be a strong indicator of quality
- Higher sweetness will negatively affect white wines quality

### Red Wine: Correlation Matrix

The matrix below shows how strongly each red wine variable correlates with others. 

#### Key Observations:

- **Alcohol** has a somewhat **positive correlation** with quality (`0.48`), suggesting higher alcohol content typically means the red wines quality is higher.
- **Volatile acidity** has a **negative correlation** (`−0.39`), which suggests higher acidity often corresponds to **lower quality**.
- **Sulphates** and **citric acid** also show positive correlations with quality.
- **Fixed acidity** and **citric acid** are positively correlated with each other — having both as indicators may be redundant.
- **Free and total sulphur dioxide** are strongly correlated with each other (`0.67`), so may also be redundant.

In [ ]:
# Define cm for both
corr_red = wine_red.corr(numeric_only=True, method='pearson')
corr_white = wine_white.corr(numeric_only=True, method='pearson')

# Red wine correlation matrix
plt.figure(figsize=(10, 8))
sb.heatmap(corr_red, annot=True, cmap="Reds", fmt=".2f")
plt.title("Red Wine Correlation Matrix")
plt.savefig("../figures/red_wine_matrix.png")
plt.show()


### White Wine: Correlation Matrix

The matrix below shows how strongly each white wine variable correlates with others. 

#### Key Observations:

- **Alcohol** again shows a **positive correlation** with quality (`0.44`), consistent with red wines.
- **Residual sugar** and **density** show **negative correlation** with quality.
- **Free sulphur dioxide** and **total sulphur dioxide** are **highly correlated** (`0.72`), similar to the red wine dataset, meaning having both here could be redundant as well.
- **Chlorides** and **density** are moderately correlated — could also be considered redundant.

In [ ]:
# White wine correlation matrix
plt.figure(figsize=(10, 8))
sb.heatmap(corr_white, annot=True, cmap="Blues", fmt=".2f")
plt.title("White Wine Correlation Matrix")
plt.savefig("../figures/white_wine_matrix.png")
plt.show()

### Task 2 Summary:

From both correlation matrix's, `alcohol` stands out as the best **positive predictor** of wine quality, while `volatile acidity` (*for red*) and `density` (*for white*) could be used as **possible negative predictors** in quality.

#### For Later Modelling:
- Use *alcohol, sulphates, citric acid*, and *pH*.
- Don't use one of `free sulphur dioxide` or `total sulphur dioxide` as matrixs suggests them to be so closely correlated one is redundant.
- `Residual sugar` may be important indirectly (as seen in sweetness analysis), but direct correlation with quality is low.
- Some may be useful only in **non linear** models, despite weak Pearson correlation (as Pearson focus on linear relationships).

## Task 3

### Modelling

Task 3 was to experiment with one or more machine learning approaches that predict wine quality from the datasets variables.

I explored **two different approaches**, with different thresholds for binary classification:

1. **Binary classification**: Have quality scores as "low" vs "high" classes based a quality threshold (I tried 3 different thresholds).
2. **Regression**: Predict the exact quality score.

I chose to use a **Random Forest** model for both tasks, as it handles non linear relationships well and provides label importance scores. I also standardised all input labels using `StandardScaler` from **sklearn**.

For binary classification, as mentioned, I tested multiple thresholds for `high` quality wine:

- `>5`: few low quality wines - harder to classify as almost all wines are high quality and therefore positively imbalanced
- `>6`: approximately balanced classes
- `>7`: few high quality samples — harder to classify as almost all wines are low quality and therefore negatively imbalanced

For each model, I split the data into training and test sets (80/20 respectively) and used **5-fold cross validation** to measure performance. This helped to avoid overfitting and gave a better estimate on how well the model learnt, rather than memorised.

### Setup

To begin modelling, I imported the necessary libraries as seen below for both the training, testing and later (Task 4) evaluation.

I also drop the quality category from the data frame, as this is what I am trying to predict, and the two category variables I defined previously (isSweet and alcohol_cat).

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)

df_r = wine_red.copy()
df_w = wine_white.copy()
x_r = df_r.drop(columns=["quality", "alcohol_cat", "isSweet"])
y_r = df_r["quality"]

x_w = df_w.drop(columns=["quality", "alcohol_cat", "isSweet"])
y_w = df_w["quality"]

### Binary classification - RED WINE

Below, the red wine dataset is selected for **binary classification** using **3** different thresholds (*>5 or 6 or 7*) for defining `high` quality.

For each threshold, the code does the following:

1. Creates a binary variable `quality_bin`:
   - `1` = **high** quality
   - `0` = **low** quality
2. Separates data into **training and test sets** as 80/20 respectively
3. Scales the features using `StandardScaler` to make sure all variables contribute during modelling equally
4. Trains a `RandomForestClassifier` on the training data
5. Evaluates model with all suggested in task description:
   - **F1-score**
   - **AUC**
   - **ROC Curve**

I then used the results to identify which threshold had the best binary classification results. For evaluation and reasoning, see Task 4.

In [ ]:
thresholds = [5, 6, 7]
f1_scores_rw = []
auc_scores_rw = []

for threshold in thresholds:
    print(f"\n########## Binary classification where HIGH QUALITY IS > {threshold} ##########")
    
    df_r["quality_bin"] = (df_r["quality"] > threshold).astype(int)
    x_r = df_r.drop(columns=["quality", "quality_bin", "alcohol_cat", "isSweet"])
    y_r = df_r["quality_bin"]

    x_train_r, x_test_r, y_train_r, y_test_r = train_test_split(
        x_r, y_r, test_size=0.2, stratify=y, random_state=42
    )

    scaler = StandardScaler()
    x_train_scaled_r = scaler.fit_transform(x_train_r)
    x_test_scaled_r = scaler.transform(x_test_r)

    clf = RandomForestClassifier(random_state=42)
    scores = cross_val_score(clf, x_train_scaled_r, y_train_r, cv=5, scoring="f1")
    mean_f1_r = scores.mean()
    f1_scores_rw.append(mean_f1_r)
    print(f"F1-score (CV): {mean_f1_r}")

    clf.fit(x_train_scaled_r, y_train_r)
    y_pred_r = clf.predict(x_test_scaled_r)

    # AUC
    y_proba_r = clf.predict_proba(x_test_scaled_r)[:, 1]
    roc_auc_r = roc_auc_score(y_test_r, y_proba_r)
    auc_scores_rw.append(roc_auc_r)
    print(f"AUC: {roc_auc_r}")

    # Plot and save ROC Curve
    fpr, tpr, _ = roc_curve(y_test_r, y_proba_r)
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"ROC Curve (AUC = {roc_auc_r})", linewidth=2)
    plt.plot([0, 1], [0, 1], linestyle="--", color="red", label="Random Guess")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve Plot (Threshold > {threshold})")
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"../figures/rw_bc_roc_curve_threshold_{threshold}.png")
    plt.show()

    df_r.drop(columns=["quality_bin"], inplace=True)

### Regression - RED WINE

As well as using binary classification, I also modelled wine quality using quality as a **continuous variable** and used **regression**, as shown below, with the red wine dataset.

This time, the code does the following:

1. Data separated into **training and test** with same 80/20 split
2. Labels **standardised** again using `StandardScaler`
3. `RandomForestRegressor` model trained to predict the `quality` exactly
4. The training (**test data**) was measured using **5 fold cross validation**, like with classification, as well with all the suggested metrics and the negative MSE metric:
    - **MSE**
    - **NMSE**
    - **RMSE**

Again, for evaluation and reasoning see Task 4.

In [ ]:
# Regression

x_train_r, x_test_r, y_train_r, y_test_r = train_test_split(
    x_r, y_r, test_size=0.2, random_state=42
)

scaler = StandardScaler()
x_train_scaled_r = scaler.fit_transform(x_train_r)
x_test_scaled_r = scaler.transform(x_test_r)

reg = RandomForestRegressor(random_state=42)
mse_r = cross_val_score(reg, x_train_scaled_r, y_train_r, cv=5, scoring="neg_mean_squared_error")

nmse_r = mse_r.mean()
mse_r = -nmse_r
rmse_r = np.sqrt(mse_r)

print(f"Results of training MSE: {mse_r}")
print(f"Results of training NMSE: {nmse_r}")
print(f"Results of training RMSE: {rmse_r}")

### Regression Test - RED WINE

The following metrics, as suggested, were used to test how well the model performed when predicting quality scores with new data:

- **NMSE**
- **MSE**
- **RMSE**
- **MAE**
- **R2**

For evaluation and reasoning, see **Task 4**.


In [ ]:
reg.fit(x_train_scaled_r, y_train_r)
y_pred_reg_r = reg.predict(x_test_scaled_r)

mse_r = mean_squared_error(y_test_r, y_pred_reg_r)
rmse_r = np.sqrt(mse_r)
mae_r = mean_absolute_error(y_test_r, y_pred_reg_r)
r2_r = r2_score(y_test_r, y_pred_reg_r)
nmse_r = -mse_r

print("######### TEST RESULTS: #########")
print(f"Mean squared error (MSE): {mse_r}")
print(f"Negative MSE (NMSE): {nmse_r}")
print(f"Root mean squared error (RMSE): {rmse_r}")
print(f"Mean absolute error (MAE): {mae_r}")
print(f"R2 score: {r2_r}")

### Binary classification - WHITE WINE

Here, I repeat the binary classification task on the **white wine dataset** with the same three thresholds.

These results allow comparison with the red wine model to understand if white wines are harder to predict, or if the model generalises across wine types.


In [ ]:
thresholds = [5, 6, 7]
f1_scores_ww = []
auc_scores_ww = []

for threshold in thresholds:
    print(f"\n########## Binary classification where HIGH QUALITY IS > {threshold} ##########")
    
    df_w["quality_bin"] = (df_w["quality"] > threshold).astype(int)
    x_w = df_w.drop(columns=["quality", "quality_bin", "alcohol_cat", "isSweet"])
    y_w = df_w["quality_bin"]

    x_train_w, x_test_w, y_train_w, y_test_w = train_test_split(
        x_w, y_w, test_size=0.2, stratify=y_w, random_state=42
    )

    scaler = StandardScaler()
    x_train_scaled_w = scaler.fit_transform(x_train_w)
    x_test_scaled_w = scaler.transform(x_test_w)

    clf = RandomForestClassifier(random_state=42)
    scores = cross_val_score(clf, x_train_scaled_w, y_train_w, cv=5, scoring="f1")
    mean_f1_w = scores.mean()
    f1_scores_ww.append(mean_f1_w)
    print(f"F1-score (CV): {mean_f1_w}")

    clf.fit(x_train_scaled_w, y_train_w)
    y_pred_w = clf.predict(x_test_scaled_w)

    # AUC
    y_proba_w = clf.predict_proba(x_test_scaled_w)[:, 1]
    roc_auc_w = roc_auc_score(y_test_w, y_proba_w)
    auc_scores_ww.append(roc_auc_w)
    print(f"AUC: {roc_auc_w}")

    # Plot and save ROC Curve
    fpr, tpr, _ = roc_curve(y_test_w, y_proba_w)
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"ROC Curve (AUC = {roc_auc_w})", linewidth=2)
    plt.plot([0, 1], [0, 1], linestyle="--", color="orange", label="Random Guess")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve Plot (Threshold > {threshold})")
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"../figures/ww_bc_roc_curve_threshold_{threshold}.png")
    plt.show()

    df_w.drop(columns=["quality_bin"], inplace=True)

### Regression - WHITE WINE

Again as was with the red wine, I used the same metrics along with regression with the white wine dataset.

In [ ]:
# Regression
x_train_w, x_test_w, y_train_w, y_test_w = train_test_split(
    x_w, y_w, test_size=0.2, random_state=42
)

x_train_scaled_w = scaler.fit_transform(x_train_w)
x_test_scaled_w = scaler.transform(x_test_w)

mse_w = cross_val_score(reg, x_train_scaled_w, y_train_w, cv=5, scoring="neg_mean_squared_error")

nmse_w = mse_w.mean()
mse_w = -nmse_w
rmse_w = np.sqrt(mse_w)

print(f"Results of training MSE: {mse_w}")
print(f"Results of training NMSE: {nmse_w}")
print(f"Results of training RMSE: {rmse_w}")

### Regression Test - WHITE WINE

Again, with same metrics as with red wine, I tested how well the model performed.

- **NMSE**
- **MSE**
- **RMSE**
- **MAE**
- **R2**

In [ ]:
reg.fit(x_train_scaled_w, y_train_w)
y_pred_reg_w = reg.predict(x_test_scaled_w)

mse_w = mean_squared_error(y_test_w, y_pred_reg_w)
rmse_w = np.sqrt(mse_w)
mae_w = mean_absolute_error(y_test_w, y_pred_reg_w)
r2_w = r2_score(y_test_w, y_pred_reg_w)
nmse_w = -mse_w

print("######### TEST RESULTS: #########")
print(f"Mean squared error (MSE): {mse_w}")
print(f"Negative MSE (NMSE): {nmse_w}")
print(f"Root mean squared error (RMSE): {rmse_w}")
print(f"Mean absolute error (MAE): {mae_w}")
print(f"R2 score: {r2_w}")

## Task 4

### Model Evaluation

Task 4 was to evaluate the model using cross validation - I used 5 fold cross validation, which meant the model was trained on 80% of data and tested on 20%. 

The two different approaches I tried, as highlighted in Task 3, where:
1. **Binary classification**
2. **Regression**

The **metrics chosen** for my evaluation where:

*For BC* ->
   - **F1-score**
   - **AUC**
   - **ROC Curve**
*For R* ->
   - **NMSE**
   - **MSE**
   - **RMSE**
   - **MAE**
   - **R2**


### Binary Classification Review

To evaluate performance for BC, I plotted the two of the metrics suggested (**F1-score and AUC**) across the different binary classification thresholds tested (i.e. >5, >6, >7).

#### Metrics

- F1-score (Training): Measures the balance of precision and recall during training, averaged across 5-fold cross-validation.

- AUC (Testing): Shows model’s ability to separate the high vs low quality classes on the unseen test data, based on predicted probabilities.

The plot makes it easier to identify the best performing threshold and examine trade offs with each threshold. For example, what I didn't predict, that is shown, is a higher threshold (e.g. >7) gives higher AUC but a lower F1-score, showing the trade off between model confidence and balance.

#### Observations
- Threshold >6 achieves the best trade-off for both datasets, with high F1 and AUC scores — suggesting this threshold gives the most balanced classification.
- Red wine performs better at lower thresholds (e.g. >5) in terms of F1, likely due to fewer high-quality red wines making classification easier.
- White wine performs better at higher thresholds in terms of AUC, suggesting it distinguishes high-quality samples better despite more class imbalance.
- Threshold >7 leads to a sharp F1 drop, especially in red wine, due to the small number of samples labelled as "high quality", which reduces recall.

In [ ]:
x = range(3)
width = 0.2

plt.figure(figsize=(10, 6))

# Bars for Red Wine
plt.bar([i - 1.5*width for i in x], f1_scores_rw, width, label="Red F1", color="lightcoral")
plt.bar([i - 0.5*width for i in x], auc_scores_rw, width, label="Red AUC", color="indianred")

# Bars for White Wine
plt.bar([i + 0.5*width for i in x], f1_scores_ww, width, label="White F1", color="gold")
plt.bar([i + 1.5*width for i in x], auc_scores_ww, width, label="White AUC", color="orange")

plt.xticks(x, thresholds)
plt.ylabel("Score")
plt.ylim(0, 1.1)
plt.title("Binary Classification Review (Red vs White)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig("../figures/bc_comparison.png")
plt.show()

### Regression Review

To compare how well the regression model performs during training vs on unseen test data, I visualised the most relevant error metrics using a grouped bar chart.

This chart evaluates the model's **generalisation ability** — whether it's just memorising the training data or actually learning to make accurate predictions (if there is overfitting).

#### Metrics shown:
- **MSE** (Mean Squared Error): For large errors (scaled/standardised)
- **RMSE** (Root Mean Squared Error): Also for large errors but for the original scale of the target variable
- **MAE** (Mean Absolute Error): Measures average error size

#### Observation:
- For both wine types, the training and test `MSE` and `RMSE` scores are quite close, suggesting the models generalise well.
- White wine has slightly higher test `MAE`, implying that while it learns well, it makes slightly more consistent errors on unseen samples than the red model.
- There is no strong evidence of overfitting — test performance is similar to training in all cases.
- The RMSE scores (`~0.14–0.15`) suggest the model is typically off by less than 1 quality point on average.

Training scores were obtained using **5-fold cross-validation**, and test scores come from the **unseen 20% test data**.

This comparison helps reveal signs of **overfitting** (if test errors are much higher than training) or **underfitting** (if both are high).

In [ ]:
train_mse_r = mse_r
train_rmse_r = rmse_r

test_mse_r = mean_squared_error(y_test_r, y_pred_reg_r)
test_rmse_r = np.sqrt(test_mse_r)
test_mae_r = mean_absolute_error(y_test_r, y_pred_reg_r)

train_mse_w = mse_w
train_rmse_w = rmse_w

test_mse_w = mean_squared_error(y_test_w, y_pred_reg_w)
test_rmse_w = np.sqrt(test_mse_w)
test_mae_w = mean_absolute_error(y_test_w, y_pred_reg_w)

metrics = ["MSE", "RMSE", "MAE"]
train_red = [train_mse_r, train_rmse_r, 0]
test_red = [test_mse_r, test_rmse_r, test_mae_r]

train_white = [train_mse_w, train_rmse_w, 0]
test_white = [test_mse_w, test_rmse_w, test_mae_w]

width = 0.2
plt.figure(figsize=(10, 6))

# red
plt.bar([i - 1.5 * width for i in x], train_red, width, label="Red Train", color="lightcoral")
plt.bar([i - 0.5 * width for i in x], test_red, width, label="Red Test", color="indianred")

# White
plt.bar([i + 0.5 * width for i in x], train_white, width, label="White Train", color="gold")
plt.bar([i + 1.5 * width for i in x], test_white, width, label="White Test", color="orange")

plt.xticks(x, metrics)
plt.ylabel("Error")
plt.title("Regression: Red vs White and Train vs Test")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig("../figures/regression_error_comparison.png")
plt.show()

### End Task 4

In summary, to evaluate model performance, I used both **classification** and **regression** approaches and compared them using appropriate metrics.

### Binary Classification Evaluation

Binary classification models were trained using three thresholds for "high quality" (`>5`, `>6`, `>7`). Each model was evaluated using:
- **F1-score** (on training data via 5-fold cross-validation)
- **AUC** (on test set)
- **ROC Curve** (visual performance of true vs false identification of quality)

**Overall:**
- **Threshold >5** gives the highest F1-score, but performance is misleading due to **class imbalance** (very few *low quality* samples).
- **Threshold >6** provides the most balanced result, with both F1 and AUC scores being high.
- **Threshold >7** results in high AUC, but lower F1-score due to a small number of positive (high quality) samples, reducing amount of recall.

The bar chart comparing performance across thresholds shows that **threshold >6** is the best option overall.

### Regression Evaluation

I also used **Random Forest Regression** to predict the exact quality score as a continuous variable.

- Metrics used:
  - **MSE**, **RMSE**, **MAE** (on both training and test sets)
  - **5-fold cross-validation** for training scores
- A grouped bar chart to visualise training vs test errors

**Findings:**
- Training and test errors are close, indicating **good generalisation** with no major overfitting
- The regression model achieved a test **RMSE of `0.26`**, which is quite good
- **MAE** was low (`0.15`), meaning predictions are close to actual values on average.
- For both wine types, threshold >6 gave the best balance of performance across both F1 and AUC.
- Threshold >5 gave high F1-scores, but these are misleading due to class imbalance (most wines are above this score).
- Threshold >7 gave very high AUC in some cases, but the models struggled with F1-score due to the rarity of high-quality samples, especially in red wine.
- White wines generally showed higher AUC across all thresholds, suggesting they are easier to separate by quality in probability space.

---

### Conclusion

- **Binary classification** (threshold >6) offers clearer interpretability
- **Regression** gives more specific predictions but is harder to interpret
- Overall, both models performed well, with **no extreme overfitting** and good predictability


## Extensions

### 1. Additional Descriptive Analysis

To explore interactions between sweetness (`isSweet`), alcohol level (`alcohol_cat`), and wine quality, I merged red and white datasets and added a `type` column. Using boxplots grouped by `type`, I visualised how these factors influenced quality.

---

### 2. Correlation and Feature Selection

I generated a Pearson correlation matrix again, this time with both datasets combined, to assess linear relationships between features and wine quality.

**Strong predictors of quality**:
- `alcohol` had the highest positive correlation with `quality`
- `volatile acidity` had the strongest negative correlation

**Redundant pairs** (highly correlated features):
- `residual sugar` and `density` showed strong correlation — likely redundant
- In later modelling, it might be better to keep only one (e.g., `density`)

---

### Classification

Instead of binarising quality scores, I treated this as a multi-class classification problem (predicting wine quality from 3 to 9).

I trained a **Random Forest classifier** once more, still using 5-fold cross-validation, and evaluated the model on the test set using precision, recall, F1-score, and a confusion matrix.

**Key Insights**:
- The model performed best on predicting mid-quality wines (5–6), which dominate the dataset.
- Performance on extreme values (3, 9) was weaker due to class imbalance.
- Overall, the classifier captured the trends well


### E1A

**Findings:**
- Wines with **high alcohol content** stil consistently had higher median quality than low alcohol wines.
- **Sweet wines** showed slightly lower median quality than dry wines, especially among red wines.
- Red wines generally received **lower quality scores** than white wines overall.

I also compared the full distribution of quality scores across all wines, finding a concentration around 5–6 but wider spread in the white wines.


In [ ]:
wine_all = pd.concat([wine_red, wine_white], ignore_index=True)
wine_all_clean = wine_all.dropna(subset=['alcohol_cat', 'isSweet'])

# Ensure alcohol_cat is categorical with correct order
wine_all_clean['alcohol_cat'] = pd.Categorical(
    wine_all_clean['alcohol_cat'],
    categories=["low", "mid", "high"],
    ordered=True
)

# If isSweet is binary (0/1), make it a string for clarity
if wine_all_clean['isSweet'].dtype != 'object':
    wine_all_clean['isSweet'] = wine_all_clean['isSweet'].map({0: "Dry", 1: "Sweet"})

wine_red['type'] = 'red'
wine_white['type'] = 'white'
# Plot quality across alcohol_cat and isSweet
sb.catplot(
    data=wine_all,
    x="alcohol_cat",
    y="quality",
    hue="isSweet",
    col="type",
    kind="box",
    order=["low", "mid", "high"]
)
plt.suptitle("Wine Quality by Alcohol Level and Sweetness", y=1.05)
plt.savefig("../figures/wine_quality_alco_sweetness")
plt.show()

### E1B

In [ ]:
# Boxplot comparing quality across wine types
plt.figure(figsize=(6, 5))
sb.boxplot(data=wine_all, x="type", y="quality")
plt.title("Quality Distribution: Red vs White Wine")
plt.savefig("../figures/quality_dis_red_v_white")
plt.show()

### E1C

In [ ]:
# Overall quality distribution regardless of type
plt.figure(figsize=(6, 5))
sb.countplot(data=wine_all, x="quality")
plt.title("Overall Wine Quality Distribution")
plt.xlabel("Quality Score")
plt.ylabel("Count")
plt.savefig("../figures/overall_quality_distribution")
plt.show()


### E2

In [ ]:
# Compute Pearson correlation matrix
corr_matrix = wine_all.corr(numeric_only=True, method='pearson')

# Plot correlation heatmap
plt.figure(figsize=(12, 10))
sb.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Pearson Correlation Matrix")
plt.savefig("../figures/both_matrix")
plt.show()

### E3

In [ ]:
# Drop non-numeric or non-useful columns for modeling
features_to_drop = ['quality', 'alcohol_cat', 'isSweet', 'type']  # drop target + non-numeric
X = wine_all.drop(columns=features_to_drop, errors='ignore')
y = wine_all['quality']

# Optional: scale features (not strictly required for trees)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sb
import matplotlib.pyplot as plt

# Predict
y_pred = rf.predict(X_test)

# Report
print(classification_report(y_test, y_pred))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sb.heatmap(cm, annot=True, fmt='d', cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Multi-class Quality Prediction")
plt.savefig("../figures/confusion_matrix")
plt.show()

## References

https://github.com/rhiever/Data-Analysis-and-Machine-Learning-Projects/blob/master/example-data-science-notebook/Example%20Machine%20Learning%20Notebook.ipynb

https://github.com/PacktPublishing/Jupyter-Notebook-for-Data-Science

https://nbviewer.org/github/Tanu-N-Prabhu/Python/blob/master/Data_Cleaning/Data_Cleaning_using_Python_with_Pandas_Library.ipynb

https://www.markdownguide.org/cheat-sheet/

